# Tamil Nadu 9-BA rolling-year results analysis

This notebook analyses the full FY2025–26 rolling unit-commitment run. It reads only the seven-day retained portions of each checkpoint; eighth-day look-ahead results are intentionally excluded.

It validates chronological coverage and weekly state transfer, compares modeled generation with observed annual and monthly generation, calculates full-load hours (FLH), tests whether wind availability and hydro utilization reproduce the observed seasonal timing, and examines dispatch, commitment, storage, imports, unserved energy, ramps, and transmission loading.

In [ ]:
# ===================== USER SETTINGS =====================
ROLLING_RESULT_DIR = None  # None uses the standard FY2025-26 output folder
REQUIRE_COMPLETE_YEAR = True
SAVE_TABLES_AND_FIGURES = True
# =========================================================

## Load and validate retained weekly checkpoints

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent
if not (ROOT / 'scripts').exists():
    raise FileNotFoundError('Run this notebook from the repository or 9_BA folder.')
sys.path.insert(0, str(ROOT / 'scripts'))
from run_9ba_weekly_days_2025_26 import read_observed_generation

RESULT_DIR = (
    Path(ROLLING_RESULT_DIR)
    if ROLLING_RESULT_DIR is not None
    else ROOT / '9_BA' / 'run_results' / 'rolling_2025-04-01_2026-03-31'
)
CHECKPOINT_DIR = RESULT_DIR / 'weekly_networks'
ANALYSIS_DIR = RESULT_DIR / 'analysis'
files = sorted(CHECKPOINT_DIR.glob('window_*.nc'))
if not files:
    raise FileNotFoundError(f'No rolling checkpoints found in {CHECKPOINT_DIR}')
networks = [pypsa.Network(path) for path in files]
snapshots = pd.DatetimeIndex(np.concatenate([n.snapshots.values for n in networks]))
expected = pd.date_range('2025-04-01 00:00', '2026-03-31 23:00', freq='h')
assert snapshots.is_monotonic_increasing and not snapshots.has_duplicates
assert (snapshots.to_series().diff().dropna() == pd.Timedelta(hours=1)).all()
complete = snapshots.equals(expected)
if REQUIRE_COMPLETE_YEAR and not complete:
    missing = expected.difference(snapshots)
    raise RuntimeError(f'Rolling run is incomplete: {len(snapshots):,}/8,760 hours; {len(missing):,} missing.')
print(f'Loaded {len(files)} retained checkpoints from {RESULT_DIR}')
print(f'Coverage: {snapshots[0]} through {snapshots[-1]} ({len(snapshots):,} hours)')
print(f'Complete FY2025-26: {complete}')

In [ ]:
base = networks[0]
def combine(attribute, table):
    return pd.concat([getattr(getattr(n, attribute), table) for n in networks]).sort_index()

generator_p = combine('generators_t', 'p')
generator_p_max_pu = combine('generators_t', 'p_max_pu')
generator_status = combine('generators_t', 'status')
generator_start_up = combine('generators_t', 'start_up')
generator_shut_down = combine('generators_t', 'shut_down')
storage_p = combine('storage_units_t', 'p')
storage_inflow = combine('storage_units_t', 'inflow')
storage_soc = combine('storage_units_t', 'state_of_charge')
link_p0 = combine('links_t', 'p0')
load = combine('loads_t', 'p')
weights = pd.concat([n.snapshot_weightings.generators for n in networks]).sort_index()
for frame in [generator_p, generator_p_max_pu, generator_status, generator_start_up, generator_shut_down, storage_p, storage_inflow, storage_soc, link_p0, load]:
    assert frame.index.equals(snapshots)
assert weights.index.equals(snapshots) and np.allclose(weights, 1.0)
print(f'Combined {generator_p.shape[1]} generators, {storage_p.shape[1]} storage units, and {link_p0.shape[1]} corridors.')

## Weekly boundary-condition audit

For every adjacent checkpoint, this verifies storage state of charge, online-generator preceding dispatch, and consecutive on/off duration transferred into the next solve.

In [ ]:
boundary_rows = []
committable = base.generators.index[base.generators.committable]
for prior, current in zip(networks[:-1], networks[1:]):
    prior_status = prior.generators_t.status[committable].round().astype(int)
    final_status = prior_status.iloc[-1]
    expected_run = {}
    for unit in committable:
        value = int(final_status[unit])
        count = 0
        for item in prior_status[unit].to_numpy()[::-1]:
            if int(item) != value:
                break
            count += 1
        expected_run[unit] = count
    expected_run = pd.Series(expected_run)
    expected_up = expected_run.where(final_status.eq(1), 0)
    expected_down = expected_run.where(final_status.eq(0), 0)
    online = final_status.index[final_status.eq(1)]
    dispatch_error = (
        current.generators.loc[online, 'p_init']
        - prior.generators_t.p.iloc[-1].reindex(online)
    ).abs().max()
    soc_error = (
        current.storage_units.state_of_charge_initial
        - prior.storage_units_t.state_of_charge.iloc[-1]
    ).abs().max() if len(current.storage_units) else 0.0
    boundary_rows.append({
        'next_window_start': current.snapshots[0],
        'status_history_matches': bool(
            np.allclose(current.generators.loc[committable, 'up_time_before'], expected_up)
            and np.allclose(current.generators.loc[committable, 'down_time_before'], expected_down)
        ),
        'max_online_p_init_error_mw': float(dispatch_error) if len(online) else 0.0,
        'max_storage_initial_soc_error_mwh': float(soc_error),
    })
boundary_audit = pd.DataFrame(boundary_rows)
display(boundary_audit.head())
display(pd.Series({
    'boundaries_checked': len(boundary_audit),
    'all_status_histories_match': boundary_audit.status_history_matches.all(),
    'maximum_dispatch_handoff_error_mw': boundary_audit.max_online_p_init_error_mw.max(),
    'maximum_storage_handoff_error_mwh': boundary_audit.max_storage_initial_soc_error_mwh.max(),
}, name='value').to_frame())

## Annual generation and full-load hours

FLH is annual generation (MWh) divided by installed power capacity (MW). Modeled and observed FLH use the same model capacity denominator so they are directly comparable. These observed values are therefore **implied FLH**, not independently reported plant FLH. The model fleet contains a 31 July 2026 capacity overlay while generation covers FY2025–26; interpret both FLH columns with that vintage limitation. Hydro generation and capacity both include pumped storage.

In [ ]:
observed_daily = read_observed_generation().reindex(pd.date_range('2025-04-01', '2026-03-31', freq='D'))
carrier_hourly_mw = generator_p.T.groupby(base.generators.carrier).sum().T
generator_mwh = generator_p.mul(weights, axis=0).sum().groupby(base.generators.carrier).sum()
storage_discharge_mwh = storage_p.clip(lower=0).mul(weights, axis=0).sum()
storage_discharge_by_carrier = storage_discharge_mwh.groupby(base.storage_units.carrier).sum()
technologies = ['coal', 'oil_gas', 'nuclear', 'hydro', 'solar', 'wind']
model_mwh = pd.Series({c: float(generator_mwh.get(c, 0.0)) for c in technologies})
model_mwh['hydro'] += float(storage_discharge_by_carrier.get('hydro', 0.0))
observed_mwh = observed_daily[technologies].sum() * 1_000.0
capacity_mw = base.generators.groupby('carrier').p_nom.sum().reindex(technologies).fillna(0.0)
capacity_mw['hydro'] += base.storage_units.loc[base.storage_units.carrier.eq('hydro'), 'p_nom'].sum()
annual_comparison = pd.DataFrame({
    'installed_capacity_mw': capacity_mw,
    'modeled_generation_gwh': model_mwh / 1_000.0,
    'observed_generation_gwh': observed_mwh / 1_000.0,
})
annual_comparison['difference_gwh'] = annual_comparison.modeled_generation_gwh - annual_comparison.observed_generation_gwh
annual_comparison['difference_pct'] = 100 * annual_comparison.difference_gwh / annual_comparison.observed_generation_gwh.replace(0, np.nan)
annual_comparison['modeled_flh'] = model_mwh / capacity_mw
annual_comparison['observed_implied_flh'] = observed_mwh / capacity_mw
annual_comparison['flh_difference'] = annual_comparison.modeled_flh - annual_comparison.observed_implied_flh
annual_comparison.index.name = 'technology'
display(annual_comparison.round(1))
print(f"Modeled comparable total: {annual_comparison.modeled_generation_gwh.sum():,.1f} GWh")
print(f"Observed comparable total: {annual_comparison.observed_generation_gwh.sum():,.1f} GWh")

In [ ]:
labels = ['Coal', 'Oil & gas', 'Nuclear', 'Hydro', 'Solar', 'Wind']
x = np.arange(len(technologies)); width = 0.38
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(x - width/2, annual_comparison.modeled_generation_gwh, width, label='Modeled')
axes[0].bar(x + width/2, annual_comparison.observed_generation_gwh, width, label='Observed')
axes[0].set_ylabel('Annual generation (GWh)'); axes[0].set_title('FY2025–26 generation')
axes[1].bar(x - width/2, annual_comparison.modeled_flh, width, label='Modeled')
axes[1].bar(x + width/2, annual_comparison.observed_implied_flh, width, label='Observed implied')
axes[1].set_ylabel('Full-load hours (h/year)'); axes[1].set_title('FLH on common capacity basis')
for ax in axes:
    ax.set_xticks(x, labels, rotation=30, ha='right'); ax.grid(axis='y', alpha=.25); ax.legend()
fig.tight_layout(); annual_figure = fig

## Monthly modeled-versus-observed generation

In [ ]:
storage_hourly_mw = storage_p.clip(lower=0).sum(axis=1)
modeled_daily = carrier_hourly_mw.reindex(columns=technologies, fill_value=0.0).copy()
modeled_daily['hydro'] += storage_hourly_mw
modeled_daily = modeled_daily.mul(weights, axis=0).resample('D').sum() / 1_000.0
monthly_modeled = modeled_daily.resample('MS').sum()
monthly_observed = observed_daily[technologies].resample('MS').sum()
fig, axes = plt.subplots(3, 2, figsize=(15, 11), sharex=True)
for ax, carrier, label in zip(axes.flat, technologies, labels):
    ax.plot(monthly_modeled.index, monthly_modeled[carrier], marker='o', label='Modeled')
    ax.plot(monthly_observed.index, monthly_observed[carrier], marker='o', label='Observed')
    ax.set_title(label); ax.set_ylabel('GWh/month'); ax.grid(alpha=.25)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
axes[0, 0].legend(); fig.suptitle('Monthly generation: rolling model versus observed')
fig.tight_layout(rect=(0, 0, 1, .97)); monthly_figure = fig

## Wind-resource and hydro-utilization seasonality

This section separates resource availability from economic dispatch. Wind availability is the capacity-weighted `p_max_pu` across all wind generators. Modeled and observed utilization are expressed as capacity factors on the same installed-capacity denominator. Hydro output includes conventional reservoir discharge plus pumped-storage discharge, while the separate reservoir-inflow line shows the exogenous natural-inflow proxy on the conventional reservoir capacity denominator.

The diagnostics test seasonal amplitude, peak/trough timing, monthly-pattern correlation, and overlap between the three lowest-utilization months. A flat wind-availability line alongside a seasonal observed line is direct evidence that the resource profile cannot represent the observed monsoon cycle.

In [ ]:
def available_power_by_carrier(carrier):
    assets = base.generators.index[base.generators.carrier.eq(carrier)]
    profile = generator_p_max_pu.reindex(index=snapshots, columns=assets)
    static_limits = base.generators.loc[assets, 'p_max_pu'].to_dict()
    profile = profile.fillna(value=static_limits)
    return profile.mul(base.generators.loc[assets, 'p_nom'], axis=1).sum(axis=1)

monthly_hours = pd.Series(1.0, index=snapshots).resample('MS').sum()
wind_available_mw = available_power_by_carrier('wind')
wind_available_gwh = wind_available_mw.resample('MS').sum() / 1_000.0
wind_capacity = capacity_mw['wind']
hydro_capacity = capacity_mw['hydro']
reservoir_units = base.storage_units.index[base.storage_units.p_min_pu.ge(0.0)]
reservoir_capacity = base.storage_units.loc[reservoir_units, 'p_nom'].sum()
reservoir_inflow_mw = storage_inflow.reindex(columns=reservoir_units, fill_value=0.0).sum(axis=1)

monthly_seasonality = pd.DataFrame(index=monthly_modeled.index)
monthly_seasonality.index.name = 'month'
monthly_seasonality['wind_resource_cf'] = wind_available_gwh * 1_000 / (wind_capacity * monthly_hours)
monthly_seasonality['wind_modeled_cf'] = monthly_modeled['wind'] * 1_000 / (wind_capacity * monthly_hours)
monthly_seasonality['wind_observed_cf'] = monthly_observed['wind'] * 1_000 / (wind_capacity * monthly_hours)
monthly_seasonality['wind_curtailment_pct'] = 100 * (1 - monthly_modeled['wind'] / wind_available_gwh.replace(0, np.nan))
monthly_seasonality['hydro_modeled_cf'] = monthly_modeled['hydro'] * 1_000 / (hydro_capacity * monthly_hours)
monthly_seasonality['hydro_observed_cf'] = monthly_observed['hydro'] * 1_000 / (hydro_capacity * monthly_hours)
monthly_seasonality['hydro_reservoir_inflow_cf'] = (
    reservoir_inflow_mw.resample('MS').sum() / (reservoir_capacity * monthly_hours)
)
monthly_seasonality['wind_error_gwh'] = monthly_modeled['wind'] - monthly_observed['wind']
monthly_seasonality['hydro_error_gwh'] = monthly_modeled['hydro'] - monthly_observed['hydro']

def seasonal_summary(series):
    clean = series.dropna()
    return pd.Series({
        'mean_monthly_cf_pct': 100 * clean.mean(),
        'minimum_cf_pct': 100 * clean.min(),
        'minimum_month': clean.idxmin().strftime('%b-%Y'),
        'maximum_cf_pct': 100 * clean.max(),
        'maximum_month': clean.idxmax().strftime('%b-%Y'),
        'seasonal_range_percentage_points': 100 * (clean.max() - clean.min()),
        'coefficient_of_variation': clean.std(ddof=0) / clean.mean(),
        'peak_to_trough_ratio': clean.max() / clean.min() if clean.min() > 0 else np.inf,
    })

seasonality_summary = pd.DataFrame({
    'wind_resource': seasonal_summary(monthly_seasonality['wind_resource_cf']),
    'wind_modeled': seasonal_summary(monthly_seasonality['wind_modeled_cf']),
    'wind_observed': seasonal_summary(monthly_seasonality['wind_observed_cf']),
    'hydro_modeled': seasonal_summary(monthly_seasonality['hydro_modeled_cf']),
    'hydro_observed': seasonal_summary(monthly_seasonality['hydro_observed_cf']),
}).T

def bottom_months(series, count=3):
    return set(series.nsmallest(count).index.strftime('%b-%Y'))

wind_low_model = bottom_months(monthly_seasonality['wind_modeled_cf'])
wind_low_observed = bottom_months(monthly_seasonality['wind_observed_cf'])
hydro_low_model = bottom_months(monthly_seasonality['hydro_modeled_cf'])
hydro_low_observed = bottom_months(monthly_seasonality['hydro_observed_cf'])
seasonality_diagnostics = pd.DataFrame({
    'metric': [
        'monthly_pattern_correlation', 'three_lowest_months_modeled',
        'three_lowest_months_observed', 'low_month_overlap_count',
        'mean_absolute_monthly_error_gwh'
    ],
    'wind': [
        monthly_seasonality.wind_modeled_cf.corr(monthly_seasonality.wind_observed_cf),
        ', '.join(sorted(wind_low_model)), ', '.join(sorted(wind_low_observed)),
        len(wind_low_model & wind_low_observed), monthly_seasonality.wind_error_gwh.abs().mean(),
    ],
    'hydro': [
        monthly_seasonality.hydro_modeled_cf.corr(monthly_seasonality.hydro_observed_cf),
        ', '.join(sorted(hydro_low_model)), ', '.join(sorted(hydro_low_observed)),
        len(hydro_low_model & hydro_low_observed), monthly_seasonality.hydro_error_gwh.abs().mean(),
    ],
}).set_index('metric')

def seasonality_fit_row(candidate, observed, candidate_low, observed_low):
    observed_range = observed.max() - observed.min()
    return pd.Series({
        'mean_bias_percentage_points': 100 * (candidate.mean() - observed.mean()),
        'seasonal_amplitude_captured_pct': 100 * (candidate.max() - candidate.min()) / observed_range,
        'monthly_pattern_correlation': candidate.corr(observed),
        'peak_month_matches': candidate.idxmax().month == observed.idxmax().month,
        'trough_month_matches': candidate.idxmin().month == observed.idxmin().month,
        'three_lowest_month_overlap': len(candidate_low & observed_low),
    })

seasonality_fit = pd.DataFrame({
    'wind_resource_vs_observed': seasonality_fit_row(
        monthly_seasonality.wind_resource_cf, monthly_seasonality.wind_observed_cf,
        wind_low_model, wind_low_observed),
    'hydro_model_vs_observed': seasonality_fit_row(
        monthly_seasonality.hydro_modeled_cf, monthly_seasonality.hydro_observed_cf,
        hydro_low_model, hydro_low_observed),
}).T

display(monthly_seasonality.round(3))
display(seasonality_summary)
display(seasonality_diagnostics)
display(seasonality_fit.round(3))
print(f"Wind availability captures {seasonality_fit.loc['wind_resource_vs_observed', 'seasonal_amplitude_captured_pct']:.1f}% of the observed monthly CF range.")
print(f"Hydro dispatch captures {seasonality_fit.loc['hydro_model_vs_observed', 'seasonal_amplitude_captured_pct']:.1f}% of the observed monthly utilization range.")

In [ ]:
daily_wind_cf = pd.DataFrame({
    'Resource availability': wind_available_mw.resample('D').mean() / wind_capacity,
    'Modeled generation': modeled_daily['wind'] * 1_000 / (wind_capacity * 24),
    'Observed generation': observed_daily['wind'] * 1_000 / (wind_capacity * 24),
}).rolling(30, center=True, min_periods=15).mean() * 100
daily_hydro_cf = pd.DataFrame({
    'Reservoir inflow': reservoir_inflow_mw.resample('D').mean() / reservoir_capacity,
    'Modeled generation': modeled_daily['hydro'] * 1_000 / (hydro_capacity * 24),
    'Observed generation': observed_daily['hydro'] * 1_000 / (hydro_capacity * 24),
}).rolling(30, center=True, min_periods=15).mean() * 100

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)
wind_monthly_pct = (monthly_seasonality[['wind_resource_cf', 'wind_modeled_cf', 'wind_observed_cf']] * 100).rename(columns={
    'wind_resource_cf': 'Resource availability', 'wind_modeled_cf': 'Modeled generation',
    'wind_observed_cf': 'Observed generation',
})
for column in wind_monthly_pct:
    axes[0, 0].plot(wind_monthly_pct.index, wind_monthly_pct[column], marker='o', label=column)
axes[0, 0].legend()
axes[0, 0].set_title('Wind monthly capacity factor'); axes[0, 0].set_ylabel('%')
daily_wind_cf.plot(ax=axes[1, 0])
axes[1, 0].set_title('Wind seasonality (30-day rolling mean)'); axes[1, 0].set_ylabel('%')
hydro_monthly_pct = (monthly_seasonality[['hydro_reservoir_inflow_cf', 'hydro_modeled_cf', 'hydro_observed_cf']] * 100).rename(columns={
    'hydro_reservoir_inflow_cf': 'Reservoir inflow', 'hydro_modeled_cf': 'Modeled generation',
    'hydro_observed_cf': 'Observed generation',
})
for column in hydro_monthly_pct:
    axes[0, 1].plot(hydro_monthly_pct.index, hydro_monthly_pct[column], marker='o', label=column)
axes[0, 1].legend()
axes[0, 1].set_title('Hydro monthly utilization'); axes[0, 1].set_ylabel('Capacity factor (%)')
daily_hydro_cf.plot(ax=axes[1, 1])
axes[1, 1].set_title('Hydro utilization (30-day rolling mean)'); axes[1, 1].set_ylabel('Capacity factor (%)')
for ax in axes.flat:
    ax.grid(alpha=.25); ax.set_xlim(pd.Timestamp('2025-04-01'), pd.Timestamp('2026-03-31'))
    ax.xaxis.set_major_locator(mdates.MonthLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
fig.suptitle('FY2025-26 resource and utilization seasonality')
fig.tight_layout(rect=(0, 0, 1, .97)); seasonality_figure = fig

## Whole-year operations, commitment, storage, and adequacy

In [ ]:
total_load = load.sum(axis=1)
renewables = carrier_hourly_mw.reindex(columns=['solar', 'wind'], fill_value=0).sum(axis=1)
thermal = carrier_hourly_mw.reindex(columns=['coal', 'oil_gas', 'nuclear', 'diesel'], fill_value=0).sum(axis=1)
imports = carrier_hourly_mw.get('market_import', pd.Series(0.0, index=snapshots))
unserved = carrier_hourly_mw.get('unserved_energy', pd.Series(0.0, index=snapshots))
storage_net = storage_p.sum(axis=1)
monthly_operations = pd.DataFrame({
    'demand_gwh': total_load.resample('MS').sum()/1000,
    'renewable_gwh': renewables.resample('MS').sum()/1000,
    'thermal_nuclear_gwh': thermal.resample('MS').sum()/1000,
    'imports_gwh': imports.resample('MS').sum()/1000,
    'storage_net_output_gwh': storage_net.resample('MS').sum()/1000,
    'unserved_gwh': unserved.resample('MS').sum()/1000,
})
display(monthly_operations.round(2))
fig, axes = plt.subplots(3, 1, figsize=(15, 11), sharex=False)
monthly_operations[['renewable_gwh', 'thermal_nuclear_gwh', 'imports_gwh']].plot.bar(stacked=True, ax=axes[0])
axes[0].set_title('Monthly supply mix'); axes[0].set_ylabel('GWh'); axes[0].legend(ncol=3)
duration = pd.DataFrame({'Demand': total_load.sort_values(ascending=False).to_numpy(), 'Net load': (total_load-renewables).sort_values(ascending=False).to_numpy()})
duration.plot(ax=axes[1]); axes[1].set_title('Annual load-duration curves'); axes[1].set_ylabel('MW'); axes[1].set_xlabel('Ranked hour')
storage_soc.plot(ax=axes[2]); axes[2].set_title('Pumped-storage state of charge'); axes[2].set_ylabel('MWh'); axes[2].set_xlabel('Time')
for ax in axes: ax.grid(alpha=.25)
fig.tight_layout(); operations_figure = fig

In [ ]:
unit_metrics = pd.DataFrame(index=committable)
unit_metrics['carrier'] = base.generators.loc[committable, 'carrier']
unit_metrics['capacity_mw'] = base.generators.loc[committable, 'p_nom']
unit_metrics['online_hours'] = generator_status[committable].sum()
unit_metrics['startups'] = generator_start_up[committable].sum()
unit_metrics['shutdowns'] = generator_shut_down[committable].sum()
unit_metrics['generation_gwh'] = generator_p[committable].mul(weights, axis=0).sum()/1000
commitment_by_carrier = unit_metrics.groupby('carrier').agg(
    units=('carrier', 'size'), capacity_mw=('capacity_mw', 'sum'),
    unit_online_hours=('online_hours', 'sum'), startups=('startups', 'sum'),
    shutdowns=('shutdowns', 'sum'), generation_gwh=('generation_gwh', 'sum')
)
commitment_by_carrier['startups_per_unit'] = commitment_by_carrier.startups / commitment_by_carrier.units
display(commitment_by_carrier.round(2))
display(unit_metrics.sort_values('startups', ascending=False).head(20).round(2))

## Ramp, transmission, and reliability indicators

In [ ]:
net_load = total_load - renewables
link_loading = link_p0.abs().div(base.links.p_nom, axis=1) * 100
system_metrics = pd.Series({
    'peak_demand_mw': total_load.max(),
    'peak_net_load_mw': net_load.max(),
    'maximum_one_hour_net_load_increase_mw': net_load.diff().max(),
    'maximum_one_hour_net_load_decrease_mw': net_load.diff().min(),
    'market_import_gwh': imports.mul(weights).sum()/1000,
    'unserved_energy_gwh': unserved.mul(weights).sum()/1000,
    'hours_with_unserved_energy': unserved.gt(1e-6).sum(),
    'maximum_unserved_power_mw': unserved.max(),
    'maximum_corridor_loading_pct': link_loading.max().max(),
    'hours_any_corridor_at_or_above_99pct': link_loading.ge(99).any(axis=1).sum(),
}, name='value')
display(system_metrics.to_frame().round(3))
corridor_metrics = pd.DataFrame({
    'capacity_mw': base.links.p_nom,
    'max_abs_flow_mw': link_p0.abs().max(),
    'max_loading_pct': link_loading.max(),
    'hours_at_or_above_99pct': link_loading.ge(99).sum(),
}).sort_values('max_loading_pct', ascending=False)
display(corridor_metrics.round(2))

## Save analysis artifacts

In [ ]:
if SAVE_TABLES_AND_FIGURES:
    ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
    annual_comparison.to_csv(ANALYSIS_DIR / 'annual_generation_and_flh.csv')
    monthly_modeled.to_csv(ANALYSIS_DIR / 'monthly_modeled_generation.csv')
    monthly_observed.to_csv(ANALYSIS_DIR / 'monthly_observed_generation.csv')
    monthly_seasonality.to_csv(ANALYSIS_DIR / 'monthly_resource_seasonality.csv')
    seasonality_summary.to_csv(ANALYSIS_DIR / 'seasonality_summary.csv')
    seasonality_diagnostics.to_csv(ANALYSIS_DIR / 'seasonality_diagnostics.csv')
    seasonality_fit.to_csv(ANALYSIS_DIR / 'seasonality_fit.csv')
    monthly_operations.to_csv(ANALYSIS_DIR / 'monthly_operations.csv')
    commitment_by_carrier.to_csv(ANALYSIS_DIR / 'commitment_by_carrier.csv')
    unit_metrics.to_csv(ANALYSIS_DIR / 'commitment_by_unit.csv')
    corridor_metrics.to_csv(ANALYSIS_DIR / 'corridor_metrics.csv')
    boundary_audit.to_csv(ANALYSIS_DIR / 'boundary_condition_audit.csv', index=False)
    system_metrics.to_csv(ANALYSIS_DIR / 'system_metrics.csv', header=True)
    annual_figure.savefig(ANALYSIS_DIR / 'annual_generation_and_flh.png', dpi=170, bbox_inches='tight')
    monthly_figure.savefig(ANALYSIS_DIR / 'monthly_generation_comparison.png', dpi=170, bbox_inches='tight')
    seasonality_figure.savefig(ANALYSIS_DIR / 'resource_and_hydro_seasonality.png', dpi=170, bbox_inches='tight')
    operations_figure.savefig(ANALYSIS_DIR / 'annual_operations.png', dpi=170, bbox_inches='tight')
    print(f'Saved analysis tables and figures to {ANALYSIS_DIR}')